In [3]:
%pip install sentencepiece nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 31.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [nltk]2/3 [nltk]]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction
import re
import os
import random
import math
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import urllib.request
import zipfile

# 영어-스페인어 데이터 다운로드 및 압축 해제
zip_filename = "spa-eng.zip"
zip_url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
urllib.request.urlretrieve(zip_url, zip_filename)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_filename))

print("✅ 라이브러리 로드 및 데이터 다운로드 완료!")

✅ 라이브러리 로드 및 데이터 다운로드 완료!


In [5]:
extracted_folder = "./spa-eng"
file_path = os.path.join(extracted_folder, "spa.txt")

with open(file_path, "r", encoding="utf-8") as f:
    spa_eng_sentences = f.read().splitlines()

# 중복 제거
spa_eng_sentences = list(set(spa_eng_sentences))
total_sentence_count = len(spa_eng_sentences)

# 전처리 함수 (소문자 변환, 다중 공백 제거, 양끝 공백 제거)
def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r' {2,}', ' ', sentence)
    sentence = sentence.strip()
    return sentence

spa_eng_sentences = list(map(preprocess_sentence, spa_eng_sentences))

# 테스트 데이터 분리 (전체의 0.5%)
test_sentence_count = total_sentence_count // 200
train_spa_eng_sentences = spa_eng_sentences[:-test_sentence_count]
test_spa_eng_sentences = spa_eng_sentences[-test_sentence_count:]

# 탭(\t)을 기준으로 영어와 스페인어 문장 분리 함수
def split_spa_eng_sentences(spa_eng_sentences):
    spa_sentences, eng_sentences = [], []
    for spa_eng_sentence in tqdm(spa_eng_sentences, desc="문장 분리 중"):
        eng_sentence, spa_sentence = spa_eng_sentence.split('\t')
        spa_sentences.append(spa_sentence)
        eng_sentences.append(eng_sentence)
    return eng_sentences, spa_sentences

train_eng_sentences, train_spa_sentences = split_spa_eng_sentences(train_spa_eng_sentences)
test_eng_sentences, test_spa_sentences = split_spa_eng_sentences(test_spa_eng_sentences)

print(f"\n✅ 훈련 데이터 개수: {len(train_eng_sentences)}")
print(f"✅ 테스트 데이터 개수: {len(test_eng_sentences)}")

문장 분리 중:   0%|          | 0/118370 [00:00<?, ?it/s]

문장 분리 중:   0%|          | 0/594 [00:00<?, ?it/s]


✅ 훈련 데이터 개수: 118370
✅ 테스트 데이터 개수: 594


In [6]:
# 1. 토크나이저 훈련 함수
def generate_tokenizer(corpus, vocab_size, lang="spa-eng", pad_id=0, bos_id=1, eos_id=2, unk_id=3):
    file = f"./{lang}_corpus.txt"
    model = f"{lang}_spm"
    with open(file, 'w', encoding='utf-8') as f:
        for row in corpus: f.write(str(row) + '\n')
        
    spm.SentencePieceTrainer.Train(
        f'--input={file} --model_prefix={model} --vocab_size={vocab_size} '
        f'--pad_id={pad_id} --bos_id={bos_id} --eos_id={eos_id} --unk_id={unk_id}'
    )
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load(f'{model}.model')
    return tokenizer

# 두 언어 단어 사전 공유 (Vocab Size: 20,000)
VOCAB_SIZE = 20000
print("SentencePiece 모델 학습 중... (잠시만 기다려주세요 ⏳)")
tokenizer = generate_tokenizer(train_eng_sentences + train_spa_sentences, VOCAB_SIZE, 'spa-eng')

# 2. 코퍼스 토큰화 (안전한 수동 BOS/EOS 부착 적용)
def make_corpus(sentences, tokenizer):
    corpus = []
    bos_id = tokenizer.bos_id()
    eos_id = tokenizer.eos_id()
    
    for sentence in tqdm(sentences, desc="토큰화 중"):
        tokens = tokenizer.encode_as_ids(sentence)
        # LMS의 set_encode_extra_options("bos:eos")를 완벽하게 대체하는 로직
        tokens = [bos_id] + tokens + [eos_id] 
        corpus.append(tokens)
    return corpus

eng_corpus = make_corpus(train_eng_sentences, tokenizer)
spa_corpus = make_corpus(train_spa_sentences, tokenizer)

# 3. 데이터 패딩 (최대 길이 50)
MAX_LEN = 50
def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) > max_len:
            seq = seq[:max_len]
        else:
            seq = seq + [pad_value] * (max_len - len(seq))
        padded_sequences.append(seq)
    return torch.tensor(padded_sequences, dtype=torch.long)

enc_ndarray = pad_sequences_custom(eng_corpus, max_len=MAX_LEN, pad_value=0)
dec_ndarray = pad_sequences_custom(spa_corpus, max_len=MAX_LEN, pad_value=0)

# 4. 데이터로더 셋업
from torch.utils.data import TensorDataset, DataLoader
BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = TensorDataset(enc_ndarray, dec_ndarray)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

print(f"\n✅ 모델 훈련용 텐서/데이터로더 세팅 완료!")
print(f"인코더 텐서 크기(enc_ndarray): {enc_ndarray.shape}")
print(f"디코더 텐서 크기(dec_ndarray): {dec_ndarray.shape}")

SentencePiece 모델 학습 중... (잠시만 기다려주세요 ⏳)


I0000 00:00:1783913234.649519    3970 sentencepiece_trainer.cc:227] Running command: --input=./spa-eng_corpus.txt --model_prefix=spa-eng_spm --vocab_size=20000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
I0000 00:00:1783913234.650678    3970 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: ./spa-eng_corpus.txt
  input_format: 
  model_prefix: spa-eng_spm
  model_type: UNIGRAM
  vocab_size: 20000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed

토큰화 중:   0%|          | 0/118370 [00:00<?, ?it/s]

토큰화 중:   0%|          | 0/118370 [00:00<?, ?it/s]


✅ 모델 훈련용 텐서/데이터로더 세팅 완료!
인코더 텐서 크기(enc_ndarray): torch.Size([118370, 50])
디코더 텐서 크기(dec_ndarray): torch.Size([118370, 50])


In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# [1] Positional Encoding
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    return sinusoid_table

# [2] 마스크(Mask) 생성
def generate_padding_mask(seq):
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()

def generate_lookahead_mask(size):
    return torch.triu(torch.ones(size, size), diagonal=1)

def generate_masks(src, tgt):
    enc_mask = generate_padding_mask(src)
    dec_enc_mask = generate_padding_mask(src)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])
    dec_tgt_padding_mask = generate_padding_mask(tgt)
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)
    
    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)
    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)
    return enc_mask, dec_enc_mask, dec_mask

# [3] Multi-head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        QK = torch.matmul(Q, K.transpose(-1, -2))
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)
        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)
        return out, attentions

    def split_heads(self, x):
        bsz, seq_len, _ = x.size()
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)
        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)
        out, attention_weights = self.scaled_dot_product_attention(WQ_splits, WK_splits, WV_splits, mask)
        out = self.combine_heads(out)
        out = self.linear(out)
        return out, attention_weights

# [4] Position-wise Feed Forward Network
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.fc2(out)
        return out

# [5] Encoder Layer & Decoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual
        return out, enc_attn

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)
        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual
        return out, dec_attn, dec_enc_attn

# [6] Encoder & Decoder
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)
        return out, enc_attns

class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        return out, dec_attns, dec_enc_attns

# [7] Transformer 모델 조립
class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, src_vocab_size, tgt_vocab_size, pos_len, dropout=0.2, shared_fc=True, shared_emb=False):
        super(Transformer, self).__init__()
        self.d_model = float(d_model)
        
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)
            
        pos_encoding_np = positional_encoding(pos_len, d_model)
        self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))
        
        self.do = nn.Dropout(dropout)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.shared_fc = shared_fc
        
        if shared_fc:
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        seq_len = x.size(1)
        out = emb(x)
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.do(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        enc_in_emb = self.embedding(self.enc_emb, enc_in)
        dec_in_emb = self.embedding(self.dec_emb, dec_in)
        
        enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask)
        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

print("✅ Transformer 아키텍처 통합 정의 완료!")

✅ Transformer 아키텍처 통합 정의 완료!


In [8]:
import time

# [1] 주어진 하이퍼파라미터로 모델 인스턴스 생성
transformer = Transformer(
    n_layers=2, d_model=512, n_heads=8, d_ff=2048,
    src_vocab_size=VOCAB_SIZE, tgt_vocab_size=VOCAB_SIZE,
    pos_len=200, dropout=0.3, shared_fc=True, shared_emb=True
)
transformer = transformer.to(device)
d_model = 512

# [2] Learning Rate Scheduler & Optimizer 설정
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=60):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        
    def __call__(self, step):
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

learning_rate = LearningRateScheduler(d_model)
optimizer = torch.optim.Adam(
    transformer.parameters(), 
    lr=learning_rate(1), betas=(0.9, 0.98), eps=1e-9
)

# [3] 손실 함수 (Loss Function) 정의
def loss_function(real, pred):
    real = real.to(device)
    pred = pred.to(device)
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    loss_ = loss_.view(real.size())
    mask = (real != 0).float()
    loss_ = loss_ * mask
    return loss_.sum() / mask.sum()

# [4] Train Step 함수
def train_step(src, tgt, model, optimizer):
    model.train()
    optimizer.zero_grad()
    
    tgt_in = tgt[:, :-1]
    gold = tgt[:, 1:]
    
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
    
    src = src.to(device)
    tgt_in = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)
    
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
    
    loss = loss_function(gold, predictions)
    loss.backward()
    optimizer.step()
    
    return loss, enc_attns, dec_attns, dec_enc_attns

# [5] 본격적인 훈련 시작 (Epoch 3)
EPOCHS = 3
print("====== 🚀 영어-스페인어 번역 모델 훈련 시작 ======")

for epoch in range(EPOCHS):
    start_time = time.time()
    total_loss = 0.0
    dataset_count = len(train_dataloader)
    tqdm_bar = tqdm(total=dataset_count, desc=f"Epoch {epoch+1}")
    
    for batch, (src, tgt) in enumerate(train_dataloader):
        loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)
        total_loss += loss.item()
        
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)
        
    tqdm_bar.close()
    print(f"✅ Epoch {epoch+1} 완료 | 평균 Loss: {total_loss / dataset_count:.4f} | 소요 시간: {time.time()-start_time:.2f}초\n")

print("====== ✨ 훈련이 성공적으로 완료되었습니다! ======")

====== 🚀 영어-스페인어 번역 모델 훈련 시작 ======


Epoch 1:   0%|          | 0/1850 [00:00<?, ?it/s]

✅ Epoch 1 완료 | 평균 Loss: 3795.8799 | 소요 시간: 286.46초



Epoch 2:   0%|          | 0/1850 [00:00<?, ?it/s]

✅ Epoch 2 완료 | 평균 Loss: 2498.0673 | 소요 시간: 294.12초



Epoch 3:   0%|          | 0/1850 [00:00<?, ?it/s]

✅ Epoch 3 완료 | 평균 Loss: 2049.8268 | 소요 시간: 294.27초

====== ✨ 훈련이 성공적으로 완료되었습니다! ======


In [9]:
%pip install gensim

import gensim.downloader as api
import random
from tqdm.notebook import tqdm

# 2. 사전 훈련된 GloVe 임베딩 모델 로드 (약 370MB, 다운로드에 시간이 조금 걸립니다)
print("GloVe 임베딩 모델 다운로드 중... ⏳")
wv = api.load('glove-wiki-gigaword-300')
print("✅ 임베딩 모델 로드 완료!\n")

# 3. Lexical Substitution (단어 교체) 함수 정의
def lexical_sub(sentence, wv):
    tokens = sentence.split()
    # 임베딩 사전에 존재하는 유효한 단어만 필터링
    valid_tokens = [tok for tok in tokens if tok in wv]
    
    if not valid_tokens:
        return sentence # 바꿀 단어가 없으면 원본 그대로 반환
        
    # 랜덤으로 단어 하나 선택 후, 가장 유사한 단어 추출
    selected_tok = random.choice(valid_tokens)
    similar_word = wv.most_similar(selected_tok)[0][0]
    
    # 선택된 단어만 유사한 단어로 교체하여 새로운 문장 생성
    new_sentence = " ".join([similar_word if tok == selected_tok else tok for tok in tokens])
    return new_sentence

# 4. 테스트 데이터에 Augmentation 적용
new_corpus = []
print("데이터 부풀리기(Augmentation) 진행 중...")

for old_src in tqdm(test_eng_sentences):
    new_src = lexical_sub(old_src, wv)
    if new_src is not None:
        new_corpus.append(new_src) # 교체된 새로운 문장 추가
    new_corpus.append(old_src)     # 원본 문장도 함께 보존

print("\n✅ 데이터 부풀리기 완료! 결과 샘플 확인:")
for i, sent in enumerate(new_corpus[:10]):
    print(f"[{i}] {sent}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.2 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gensim]2m2/3 [gensim]
Note: you may need to restart the kernel to use updated packages.
GloVe 임베딩 모델 다운로드 중... ⏳
[===-----------------------------------------------] 6.2% 23.4/376.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=======-------------------------------------------] 14.6% 54.9/376.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 376.1/376.1MB downloaded
✅ 임베딩 모델 로드 완료!

데이터 부풀리기(Augmentation) 진행 중...


  0%|          | 0/594 [00:00<?, ?it/s]


✅ 데이터 부풀리기 완료! 결과 샘플 확인:
[0] today, i 'm feeling a lot better.
[1] today, i am feeling a lot better.
[2] i can't staying much longer.
[3] i can't stay much longer.
[4] this ball now that boy's treasure.
[5] this ball is that boy's treasure.
[6] 'd was very tired last night.
[7] i was very tired last night.
[8] i'll assisting you.
[9] i'll assist you.


In [6]:
from collections import Counter
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. 원본 타겟 데이터(A)에 <start>, <end> 토큰 추가
ans_corpus_se = [["<start>"] + ans + ["<end>"] for ans in ans_corpus]

# 2. 원본 데이터(Q+A)를 기반으로 공유 단어 사전 구축
words = ["<pad>", "<unk>", "<start>", "<end>"]
word_counts = Counter(word for doc in que_corpus + ans_corpus_se for word in doc)
words.extend([word for word, count in word_counts.items() if count > 0])

word2idx = {word: idx for idx, word in enumerate(words)}
idx2word = {idx: word for word, idx in word2idx.items()}
VOCAB_SIZE = len(word2idx)
print(f"✅ 공유 단어 사전 구축 완료! (총 단어 수: {VOCAB_SIZE}개)")

# 3. 텍스트를 숫자로 변환 (벡터화)
def text2sequence(corpus, word2idx):
    return [[word2idx.get(word, word2idx["<unk>"]) for word in doc] for doc in corpus]

enc_train_seq = text2sequence(que_corpus, word2idx)
dec_train_seq = text2sequence(ans_corpus_se, word2idx)

# 4. 데이터 패딩 (최대 길이 40)
MAX_LEN = 40
def pad_sequences(sequences, max_len, pad_value):
    padded = []
    for seq in sequences:
        if len(seq) > max_len: padded.append(seq[:max_len])
        else: padded.append(seq + [pad_value] * (max_len - len(seq)))
    return torch.tensor(padded, dtype=torch.long)

enc_train = pad_sequences(enc_train_seq, MAX_LEN, word2idx["<pad>"])
dec_train = pad_sequences(dec_train_seq, MAX_LEN, word2idx["<pad>"])

print(f"인코더 입력 크기: {enc_train.shape}")
print(f"디코더 입력 크기: {dec_train.shape}")

✅ 공유 단어 사전 구축 완료! (총 단어 수: 6237개)
인코더 입력 크기: torch.Size([7681, 40])
디코더 입력 크기: torch.Size([7681, 40])


In [7]:
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import time

# --- 트랜스포머 모델 정의 (압축본) ---
def generate_masks(src, tgt, pad_idx=0):
    enc_mask = (src == pad_idx).unsqueeze(1).unsqueeze(2).float()
    dec_enc_mask = (src == pad_idx).unsqueeze(1).unsqueeze(2).float()
    dec_tgt_padding_mask = (tgt == pad_idx).unsqueeze(1).unsqueeze(2).float()
    dec_lookahead_mask = torch.triu(torch.ones(tgt.shape[1], tgt.shape[1]), diagonal=1).unsqueeze(0).unsqueeze(1).to(tgt.device)
    dec_mask = torch.max(dec_tgt_padding_mask.to(tgt.device), dec_lookahead_mask)
    return enc_mask, dec_enc_mask, dec_mask

def positional_encoding(pos, d_model):
    def cal_angle(position, i): return position / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    def get_posi_angle_vec(position): return [cal_angle(position, i) for i in range(d_model)]
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    return sinusoid_table

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        bsz, seq_len, _ = Q.size()
        WQ = self.W_q(Q).view(bsz, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)
        WK = self.W_k(K).view(bsz, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)
        WV = self.W_v(V).view(bsz, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)
        
        scaled_qk = torch.matmul(WQ, WK.transpose(-1, -2)) / math.sqrt(self.depth)
        if mask is not None: scaled_qk = scaled_qk + (mask * -1e9)
        
        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, WV).permute(0, 2, 1, 3).contiguous().view(bsz, -1, self.d_model)
        return self.linear(out), attentions

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm = nn.LayerNorm(d_model)
        self.do = nn.Dropout(dropout)
    def forward(self, x, mask):
        out, _ = self.enc_self_attn(self.norm(x), self.norm(x), self.norm(x), mask)
        x = x + self.do(out)
        return x + self.do(self.ffn(self.norm(x)))

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, n_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm = nn.LayerNorm(d_model)
        self.do = nn.Dropout(dropout)
    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out, _ = self.dec_self_attn(self.norm(x), self.norm(x), self.norm(x), padding_mask)
        x = x + self.do(out)
        out, _ = self.enc_dec_attn(self.norm(x), enc_out, enc_out, dec_enc_mask)
        x = x + self.do(out)
        return x + self.do(self.ffn(self.norm(x)))

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, vocab_size, pos_len=200, dropout=0.2):
        super().__init__()
        self.d_model = float(d_model)
        self.emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_encoding", torch.tensor(positional_encoding(pos_len, d_model), dtype=torch.float32))
        self.encoder = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, vocab_size)
        self.fc.weight = self.emb.weight 
        self.do = nn.Dropout(dropout)

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        enc_out = self.do(self.emb(enc_in) * math.sqrt(self.d_model) + self.pos_encoding[:enc_in.size(1), :].unsqueeze(0))
        for layer in self.encoder: enc_out = layer(enc_out, enc_mask)
            
        dec_out = self.do(self.emb(dec_in) * math.sqrt(self.d_model) + self.pos_encoding[:dec_in.size(1), :].unsqueeze(0))
        for layer in self.decoder: dec_out = layer(dec_out, enc_out, dec_enc_mask, dec_mask)
            
        return self.fc(dec_out)

# --- 훈련 세팅 및 실행 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(n_layers=1, d_model=368, n_heads=8, d_ff=1024, vocab_size=VOCAB_SIZE, dropout=0.2).to(device)

BATCH_SIZE = 64
EPOCHS = 10
WARMUP_STEPS = 1000

train_dataset = TensorDataset(enc_train, dec_train)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        return (self.d_model ** -0.5) * min(step ** -0.5, step * (self.warmup_steps ** -1.5))

lr_scheduler = LearningRateScheduler(368, WARMUP_STEPS)
optimizer = torch.optim.Adam(model.parameters(), lr=lr_scheduler(1), betas=(0.9, 0.98), eps=1e-9)
loss_fn = nn.CrossEntropyLoss(ignore_index=word2idx["<pad>"])

#

In [9]:
import random
import torch
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 1. 누락되었던 챗봇 대답 생성 함수 다시 정의
def evaluate_chatbot(sentence, model):
    model.eval()
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)
    enc_in = torch.tensor([[word2idx.get(w, word2idx["<unk>"]) for w in tokens]], dtype=torch.long).to(device)
    dec_in = torch.tensor([[word2idx["<start>"]]], dtype=torch.long).to(device)
    
    result = []
    for _ in range(MAX_LEN):
        enc_mask, dec_enc_mask, dec_mask = generate_masks(enc_in, dec_in, pad_idx=word2idx["<pad>"])
        with torch.no_grad():
            preds = model(enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask)
            
        pred_idx = preds[0, -1].argmax().item()
        if pred_idx == word2idx["<end>"]:
            result.append("<end>")
            break
            
        result.append(idx2word[pred_idx])
        dec_in = torch.cat([dec_in, torch.tensor([[pred_idx]], device=device)], dim=-1)
    return " ".join(result)

# 2. BLEU 점수 계산 함수
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu(
        [reference], 
        candidate, 
        weights=weights, 
        smoothing_function=SmoothingFunction().method1
    )

# 3. 챗봇 성능 자동 평가 함수
def eval_chatbot_bleu(model, raw_questions, raw_answers, sample_size=10):
    total_score = 0.0
    sample_indices = random.sample(range(len(raw_questions)), sample_size)
    
    print(f"====== 📊 챗봇 성능 측정 (랜덤 샘플 {sample_size}개) ======\n")
    
    for idx in sample_indices:
        question = raw_questions[idx]
        real_answer = raw_answers[idx]
        
        pred_answer = evaluate_chatbot(question, model)
        
        real_tokens = mecab.morphs(preprocess_sentence(real_answer))
        pred_tokens = pred_answer.replace('<end>', '').strip().split()
        
        score = calculate_bleu(real_tokens, pred_tokens)
        total_score += score
        
        print(f"🗣️ 질문 (Q): {question}")
        print(f"✅ 실제 대답: {real_answer}")
        print(f"🤖 챗봇 대답: {pred_answer.replace('<end>', '').strip()}")
        print(f"📈 BLEU Score: {score:.4f}")
        print("-" * 50)
        
    avg_score = total_score / sample_size
    print(f"✨ {sample_size}개 샘플에 대한 평균 BLEU Score: {avg_score:.4f}")
    return avg_score

# 4. 평가 시작!
avg_bleu = eval_chatbot_bleu(model, questions, answers, sample_size=10)

====== 📊 챗봇 성능 측정 (랜덤 샘플 10개) ======

🗣️ 질문 (Q): 연애는 도대체 어떻게 하는거야
✅ 실제 대답: 마음이 시키는데로 하면 되요.
🤖 챗봇 대답: <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start>
📈 BLEU Score: 0.0000
--------------------------------------------------
🗣️ 질문 (Q): 위로가 되네
✅ 실제 대답: 위로가 되었다니 감사해요.
🤖 챗봇 대답: <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start> <start>
📈 BLEU Score: 0.0000
--------------------------------------------------
🗣️ 질문 (Q): 씁쓸하네.
✅ 실제 대답: 제가 있잖아요.
🤖 챗봇 대답: <start> <start> <sta

#  회고 재밌다. 랜덤샘플. 굿. 챗봇. 굿